In [2]:
# in a notebook or python REPL, fresh kernel:
import reconstruction_minimal as rmin
import testing
raw = testing.getTestData('middle')
hist_list = raw[0]
res = rmin.reconstruct_minimal(hist_list, testing.projectionPhis, n_samples=300)


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 300 is different from 3)

In [3]:
import numpy as np, testing
from reconstruct_geometryA_uvwt import invert_uvwt_point
# pick a random point from typical ROI:
pt = np.array([[10.0, 5.0, 2.0]])   # x,y,z mm
uvwt = testing.getXYZtoUVWT(pt)     # forward (from notebook)
x2,y2,z2 = invert_uvwt_point(uvwt[0,0], uvwt[0,1], uvwt[0,2], uvwt[0,3])
print("delta:", np.linalg.norm(np.array([10.,5.,2.]) - np.array([x2,y2,z2])))
print("uvwt:", uvwt, "reconstructed xyz:", x2,y2,z2)


delta: 53.8379126019326
uvwt: [[ 63.16666667 182.94018246 182.94018246 268.73993808]] reconstructed xyz: 63.8379126019326 5.0 1.9999999999999947


In [1]:
import reconstruct_geometryA_uvwt as rec
# ---------------------------------------------
# MAGNIFIED DEVIATIONS (50×)
# ---------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import svd

pts = np.asarray(pts)
center = pts.mean(axis=0)

# Compute PCA direction
U, S, Vt = svd((pts - center).T, full_matrices=False)
direction = U[:, 0]  # principal eigenvector

# Closest points on PCA line
proj = (pts - center) @ direction
closest = center + np.outer(proj, direction)

# Deviations perpendicular to line
deviations = pts - closest

# Magnify deviations x50 for visibility
magnified_pts = closest + deviations * 50.0

# ----------------------------------------------------------
# PLOT
# ----------------------------------------------------------
fig = plt.figure(figsize=(9,7), dpi=150)
ax = fig.add_subplot(111, projection='3d')

# Color by sampling order (same as your 3D cloud)
N = pts.shape[0]
times = np.linspace(0.0, 1.0, N)

sc = ax.scatter(
    magnified_pts[:,0],
    magnified_pts[:,1],
    magnified_pts[:,2],
    c=times,
    cmap='viridis',
    s=40,
    edgecolors='k',
    linewidths=0.3
)

# Plot PCA line for reference
ax.plot(
    [ep0[0], ep1[0]],
    [ep0[1], ep1[1]],
    [ep0[2], ep1[2]],
    color='red',
    lw=2.5,
    label='PCA segment'
)

ax.set_title("Magnified Deviations (50×)", fontsize=14)
ax.set_xlabel("X [mm]")
ax.set_ylabel("Y [mm]")
ax.set_zlabel("Z [mm]")

fig.colorbar(sc, ax=ax, pad=0.06, label='sampling order')
plt.tight_layout()
plt.show()


NameError: name 'pts' is not defined

In [3]:
import reconstruct_geometryA_tomo as tomo
import reconstruct_geometryA_uvwt as reconmod
raw = reconmod.testing.getTestData("middle")
# coarse grid example
voxel_params = dict(x_min=-150, x_max=200, nx=120,
                    y_min=-50, y_max=200, ny=120,
                    z_min=-300, z_max=100, nz=80)
res = tomo.tomographic_reconstruct(raw, voxel_params=voxel_params,
                                   vote_method="product",
                                   threshold_frac=0.12,
                                   min_cluster_size=30,
                                   debug_plot=True)
best = res['best_cluster']
if best is not None:
    print("center", best['center'])
    print("direction", best['direction'])
    print("ep0", best['ep0'])
    print("ep1", best['ep1'])
else:
    print("No cluster found; try lowering threshold or increasing grid.")


No cluster found; try lowering threshold or increasing grid.


In [2]:
res = tomo.tomographic_reconstruct(raw, voxel_params=..., vote_method="product", threshold_frac=0.12, debug_plot=True)
print("max density:", np.max(res['density']))
print("num nonzero voxels:", np.count_nonzero(res['density']>0))
print("threshold value:", 0.12 * np.max(res['density']))
# mask fraction (if debug provided)
if 'debug' in res:
    print("voxels mapped inside hist domain (flat mask count):", np.sum(res['debug']['mask']))


TypeError: 'ellipsis' object is not subscriptable